In [2]:
import pandas as pd
import os, gzip, time,requests

In [ ]:
def construct_complete_address(row):
    """
    Build a complete address string using:
      - from_address_num
      - to_address_num (if different from the from_address_num)
      - street_name and street_type, and
      - analysis_neighborhood (if available)
    """
    from_num = row.get("from_address_num")
    to_num   = str(row.get("to_address_num", "")).strip()
    street   = str(row.get("street_name", "")).strip()
    st_type  = str(row.get("street_type", "")).strip()
    neighborhood = str(row.get("analysis_neighborhood", "")).strip()

    if pd.isna(from_num) or pd.isna(street):
        return None


    if from_num and to_num and from_num != to_num:
        house_num = f"{from_num}-{to_num}"
    else:
        house_num = from_num
    addr = f"{house_num} {street} {st_type}".strip()
    if neighborhood:
        addr = f"{addr}, {neighborhood}"
    return addr

def process_parcel_data(input_csv, output_csv):
    # Read your CSV into a DataFrame.
    df = pd.read_csv(input_csv)
    
    # print the column names
    print("Columns in the input CSV:")
    print(df.columns.tolist())

    print(f"Total rows: {len(df)}")
    # Construct the complete address.
    df["complete_address"] = df.apply(construct_complete_address, axis=1)
    print(f"{df['complete_address'].notnull().sum()} addresses constructed.")


    # Rename the APN field from 'blklot' and select the 10 key attributes.
    top10 = df.rename(columns={"blklot": "APN"})[[
        "APN",                 # Unique Assessor Parcel Number
        "complete_address",    # Constructed complete address
        "centroid_latitude",   # Geographic latitude of the centroid
        "centroid_longitude",  # Geographic longitude of the centroid
        "active",              # Active flag
        "supdist",             # Full name of Supervisorial District
        "supname",             # Name of current Supervisor
        "police_district",     # SFPD District
        "planning_district",   # Planning district
        "data_as_of"           # Timestamp of last update in the source
    ]]
    
    # Write the selected data to an output CSV file.
    top10.to_csv(output_csv, index=False)
    print(f"Processed data written to {output_csv}")


if __name__ == "__main__":
    # Use a raw string for Windows paths.
    input_csv_file = r"Data\Parcels___Active_and_Retired_20250430.csv"
    output_csv_file = "./processed_parcels.csv"
    process_parcel_data(input_csv_file, output_csv_file)



Columns in the input CSV:
['mapblklot', 'blklot', 'block_num', 'lot_num', 'from_address_num', 'to_address_num', 'street_name', 'street_type', 'odd_even', 'in_asr_secured_roll', 'pw_recorded_map', 'zoning_code', 'zoning_district', 'date_rec_add', 'date_rec_drop', 'date_map_add', 'date_map_drop', 'date_map_alt', 'project_id_add', 'project_id_drop', 'project_id_alt', 'active', 'shape', 'centroid_latitude', 'centroid_longitude', 'supdist', 'supervisor_district', 'supdistpad', 'numbertext', 'supname', 'analysis_neighborhood', 'police_district', 'police_company', 'planning_district', 'planning_district_number', 'data_as_of', 'data_loaded_at']
Total rows: 235662
212042 addresses constructed.
Processed data written to ./processed_parcels.csv


In [22]:
#### To check if the URL for scraping will work!

##https://sanfrancisco-ca.county-taxes.com/public/search?search_query=3995156&category=gsgx_property_tax

In [ ]:
# IMPORTANT NOTE: The html files that are scraped are not useful alone since they are rendered dynamically.

# File paths and endpoint constants:
INPUT_CSV = r"Data\Parcels___Active_and_Retired_20250430.csv"
OUTPUT_DIR = r"Tax/"
BASE_URL = "https://sanfrancisco-ca.county-taxes.com/public/search"

# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create a persistent HTTP session for efficiency.
session = requests.Session()


# So the actual fix was:
# Adding a valid "Referer" header
# Including "Upgrade-Insecure-Requests": "1"

# Define headers to mimic a real browser.
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Referer": "https://sanfrancisco-ca.county-taxes.com/public",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"

}

def process_row(row):
    """
    For a given row (parcel record), check if its HTML file exists.
    If not, construct the URL (using the APN), request the HTML,
    and write it as a gzipped file.
    """
    apn = str(row["blklot"]).strip()
    output_path = os.path.join(OUTPUT_DIR, f"{apn}.html.gz")
    if os.path.exists(output_path):
        return  # Skip if the file already exists

    url = f"{BASE_URL}?search_query={apn}&category=gsgx_property_tax"
    resp = session.get(url, headers=HEADERS)
    
    if resp.status_code == 200:
        with gzip.open(output_path, "wt") as f_out:
            f_out.write(resp.text)
        print(f"APN {apn} processed successfully.")
    else:
        print(f"-> Failed for APN {apn} with status code: {resp.status_code}")
    time.sleep(1)  # Rate limiting

def main():
    df = pd.read_csv(INPUT_CSV)
    df = df.head(20)  # 👈 Only take the first 20 rows
    df.apply(process_row, axis=1)

if __name__ == "__main__":
    main()


APN 4899001 processed successfully.
APN 4901019 processed successfully.
APN 4894004 processed successfully.
APN 4894022 processed successfully.
APN 4900001 processed successfully.
APN 4900011 processed successfully.
APN 4901018 processed successfully.
APN 4894010 processed successfully.
APN 4899020 processed successfully.
APN 4902002 processed successfully.
APN 4902004 processed successfully.
APN 4902011 processed successfully.
APN 4900014 processed successfully.
APN 4900015 processed successfully.
APN 4901016 processed successfully.
APN 4902001 processed successfully.
APN 4902005 processed successfully.
APN 4902018 processed successfully.
APN 4901006 processed successfully.
APN 4902017 processed successfully.
